# Football Analytics Package — Demo Notebook

This notebook demonstrates the full capabilities of the `football-analytics` package.

We use two data sources:
- **StatsBomb open data** — player-level event data, xG, passes, defensive actions
- **football-data.org** — standings, fixtures, match results

### Sections
1. Setup and imports
2. Exploring available StatsBomb data
3. Match level analysis
4. Player comparison — key players across La Liga seasons
5. Season level analysis and comparison

### 1. Setup and imports 

In [8]:
# standard library
import warnings
warnings.filterwarnings("ignore")

# data
import pandas as pd

# our package — data layer
from football_analytics.data import FootballDataClient, StatsBombClient

# our package — analytics layer
from football_analytics.analytics import (
    # form
    get_recent_form,
    get_points_per_game,
    get_home_away_split,
    # xG
    get_match_xg_summary,
    get_player_xg_ranking,
    get_xg_overperformance,
    # standings
    get_clean_standings,
    get_expected_vs_actual,
    # player
    get_top_performers,
    add_per_90_columns,
    per_90,
)

print("imports OK")

imports OK


In [9]:
# Initialise clients 

# StatsBomb requires no API key 
sb_client = StatsBombClient()

# FootballDataClient reads the key from .env aautomatically 
fd_client = FootballDataClient()

print("Clients initialised OK")

Clients initialised OK


### 2. Exploring available StatsBomb 

First we can look at what data is contained in the open package of StatsBomb.

In [10]:
competitions = sb_client.get_competitions()
competitions

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
75,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2026-04-11T12:48:10.012987,2021-06-13T16:17:31.694,NaN,2026-04-11T12:48:10.012987
76,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2026-04-27T22:02:42.690507,2026-04-27T22:03:28.087062,2026-04-27T22:03:28.087062,2026-04-27T22:02:42.690507
77,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2026-05-05T03:03:04.199896,2026-05-05T03:05:32.480837,2026-05-05T03:05:32.480837,2026-05-05T03:03:04.199896
78,72,107,International,Women's World Cup,female,False,True,2023,2026-05-03T13:51:31.021141,2026-05-03T13:55:52.303219,2026-05-03T13:55:52.303219,2026-05-03T13:51:31.021141


In [11]:
# Let's see all the available competitions  
competitions.competition_name.unique()

<StringArray>
[          '1. Bundesliga',  'African Cup of Nations',
        'Champions League',            'Copa America',
            'Copa del Rey', 'FA Women's Super League',
      'FIFA U20 World Cup',          'FIFA World Cup',
       'Frauen Bundesliga',     'Indian Super league',
                 'La Liga',                  'Liga F',
        'Liga Profesional',                 'Ligue 1',
     'Major League Soccer',   'North American League',
                    'NWSL',          'Premier League',
                 'Serie A',           'Serie A Women',
               'UEFA Euro',      'UEFA Europa League',
       'UEFA Women's Euro',       'Women's World Cup']
Length: 24, dtype: str

In [7]:
bundesliga = competitions[
    competitions["competition_name"].str.contains("liga", case=False)
]
print(bundesliga[["competition_id", "season_id", "competition_name", "season_name"]])


    competition_id  season_id   competition_name season_name
0                9        281      1. Bundesliga   2023/2024
1                9         27      1. Bundesliga   2015/2016
38             135        281  Frauen Bundesliga   2023/2024
40              11         90            La Liga   2020/2021
41              11         42            La Liga   2019/2020
42              11          4            La Liga   2018/2019
43              11          1            La Liga   2017/2018
44              11          2            La Liga   2016/2017
45              11         27            La Liga   2015/2016
46              11         26            La Liga   2014/2015
47              11         25            La Liga   2013/2014
48              11         24            La Liga   2012/2013
49              11         23            La Liga   2011/2012
50              11         22            La Liga   2010/2011
51              11         21            La Liga   2009/2010
52              11      